# 교시별 실습 Notebook

d2-p03

수업 페이지의 순서대로 실행합니다. 정답 블록과 선택 웹 자동화는 설명을 읽고 별도로 실행하세요.

In [ ]:
from pathlib import Path
import os

candidates = [Path.cwd(), *Path.cwd().parents]
course_root = next((p for p in candidates if (p / "raw/기존강사 수업자료").is_dir()), None)
if course_root is None:
    raise FileNotFoundError("수업 자료를 풀고 raw 폴더가 있는 위치에서 Notebook을 여세요.")
os.chdir(course_root)
print("수업 자료 폴더:", Path.cwd())

## 선행 셀 준비

d2-p01-02의 기본 셀입니다. 이 Notebook을 단독으로 열어도 필요한 변수를 준비할 수 있도록 포함했습니다.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

titanic = pd.read_excel(Path("raw/기존강사 수업자료/titanic.xlsx"))
plots = Path("practice/plots")
plots.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([1, 2, 3], [2, 4, 3], marker="o")
ax.set(title="Plot anatomy", xlabel="Step", ylabel="Value")
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
sex_rates = titanic.groupby("sex")["survived"].agg(["count", "mean"])
print(sex_rates.round(4))
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(sex_rates.index, sex_rates["mean"] * 100)
ax.bar_label(bars, fmt="%.1f%%")
ax.set(title="Survival rate by sex", xlabel="Sex", ylabel="Survival rate (%)", ylim=(0, 100))
fig.tight_layout()
fig.savefig(plots / "survival_by_sex.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
known_age = titanic.dropna(subset=["age"])
print("그림에 포함된 승객:", len(known_age), "나이 미상:", titanic["age"].isna().sum())
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=known_age, x="age", bins=20, ax=ax)
ax.set(title="Age distribution (known ages only)", xlabel="Age", ylabel="Passenger count")
fig.tight_layout()
fig.savefig(plots / "age_histogram.png", dpi=150)
plt.show()
plt.close(fig)

In [ ]:
columns = ["pclass", "age", "sibsp", "parch", "fare", "survived"]
corr = titanic[columns].corr()
print(corr.round(2))
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, center=0, ax=ax)
ax.set_title("Numeric correlations")
fig.tight_layout()
fig.savefig(plots / "correlation_heatmap.png", dpi=150)
plt.show()
plt.close(fig)

# 요약

성별과 객실 등급을 함께 나누어 인원·생존율을 비교하고 관찰, 해석, 추가 질문을 구분합니다.

## 2일차 3교시 — 그림에서 근거 있는 문장 만들기

2026년 10월 7일(수) · 2일차 3교시.

## 시작 전에

1~2교시의 `day2_visualization` Notebook을 이어 씁니다. `titanic`, `plots`, `plt`가 필요합니다. Kernel을 다시 시작했다면 1~2교시 준비 셀을 먼저 실행합니다. 새 Notebook에서는 아래 준비 셀부터 실행할 수 있습니다.

코드 블록 하나를 Notebook의 코드 셀 하나에 붙여 넣고 위에서 아래로 Shift+Enter로 실행합니다. 앞 셀을 실행하지 않고 뒤 셀부터 실행하면 변수를 찾지 못할 수 있습니다. 파일 경로는 자료 폴더의 최상위 위치(`raw`가 있는 폴더)를 기준으로 합니다. 현재 위치가 다르면 파일 경로를 실제 파일 위치로 바꿉니다.

## 교시별 진행

- 이전 그래프에서 축·표본 수·결측 처리 기준 찾기 → 인사이트 문장 작성 → 성별×등급별 비교 → 짝과 해석 검토 → 확인 문제.

## 준비 셀

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

titanic = pd.read_excel(Path("raw/기존강사 수업자료/titanic.xlsx"))
plots = Path("practice/plots")
plots.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot([1, 2, 3], [2, 4, 3], marker="o")
ax.set(title="Plot anatomy", xlabel="Step", ylabel="Value")
fig.tight_layout()
plt.show()
plt.close(fig)

## 3. 데이터 패턴 기반 인사이트 도출

그림을 읽을 때는 “무엇을 계산했는가 → 어떤 차이가 보이는가 → 얼마나 확실한가” 순서로 설명합니다.

| 질문 | 적절한 그림 | 결과를 설명할 때 포함할 내용 |
| --- | --- | --- |
| 성별 생존율이 다른가? | 비율 막대그래프 | 그룹별 인원, 생존율, 관찰된 차이 |
| 어떤 나이가 많은가? | 히스토그램 | 구간 수, 나이 미상 제외 기준 |
| 수치 변수 관계가 있는가? | 상관계수 히트맵 | 변수 종류, 결측 처리, 인과 해석의 한계 |

예시 문장: “제공 자료에서 여성의 생존율은 남성보다 높았다. 여성 466명과 남성 843명을 비교한 관찰 결과이며, 객실 등급 등 다른 조건을 통제한 원인 효과는 아니다.”

**종합 실습:** 성별과 객실 등급을 함께 나눠 생존율을 비교합니다. 6개 그룹의 인원 표와 그래프를 만든 뒤 “관찰 1개, 추가 확인할 질문 1개”를 씁니다.

<details>
<summary>종합 실습 예시 코드</summary>

```python
grouped = titanic.groupby(["sex", "pclass"])["survived"].agg(["count", "mean"])
print(grouped.round(4))
wide = grouped["mean"].unstack("pclass") * 100
ax = wide.plot.bar(figsize=(8, 4), ylim=(0, 100), rot=0)
ax.set(title="Survival by sex and class", xlabel="Sex", ylabel="Survival rate (%)")
ax.figure.tight_layout()
ax.figure.savefig(plots / "survival_by_sex_and_class.png", dpi=150)
plt.show()
plt.close(ax.figure)
```

</details>

## 확인 문제

1. 생존자 수가 많은 그룹이 생존율도 높은가요?
2. 나이 결측치를 0으로 채우면 어떤 막대가 왜곡될까요?
3. 상관계수 0에 가까우면 모든 종류의 관계가 없다고 말할 수 있나요?

<details>
<summary>정답</summary>

1. 그렇지 않습니다. 생존자 수를 그룹 전체 인원으로 나눠 비교해야 합니다.
2. 0세 부근의 빈도가 늘어나 실제 영유아가 많은 것으로 보일 수 있습니다.
3. 아닙니다. 선형 상관이 약하다는 뜻이며 비선형 관계나 다른 변수의 영향을 더 살펴야 합니다.

</details>

## 직접 해보기와 정답

같은 결과를 다음 세 줄로 작성합니다.

1. 관찰: 어떤 그룹에서 어떤 값이 더 높았는가?
2. 근거: 비교한 그룹의 인원과 계산 방법은 무엇인가?
3. 한계·추가 질문: 이 그림만으로 무엇을 확정할 수 없고, 무엇을 더 확인할 것인가?

<details>
<summary>정답 예시와 자기 점검</summary>

“제공 자료에서는 여성 생존율이 남성 생존율보다 높다. 전체 성별 비교의 분모는 여성 466명·남성 843명이고, survived 평균을 비율로 계산했다. 등급 등 다른 조건의 영향을 통제하지 않았으므로 성별의 원인 효과라고 확정하지 않는다. 같은 등급 안에서도 차이가 나타나는지 추가로 확인한다.”

숫자는 직접 출력한 표에서 확인해 적습니다. “여성이기 때문에 생존했다”처럼 원인을 단정하거나, 인원이 적은 그룹의 비율을 전체 모집단의 확정된 수치처럼 적지 않습니다.

</details>

## 흔한 오류와 해결

| 증상·오해 | 해결 |
| --- | --- |
| NameError: titanic | 준비 셀을 실행합니다. |
| 생존자 수로 생존율을 설명함 | sum과 count를 구분하고 mean 또는 sum/count를 씁니다. |
| 색이 진해서 중요한 변수라고 생각함 | 색 범위·상관계수·변수 종류를 확인합니다. |
| 상관계수로 인과관계를 주장함 | 관찰 문장으로 고치고 다른 조건을 확인할 질문을 씁니다. |

## 완료 체크

- [ ] 수업 폴더에서 uv로 Notebook을 열고 Kernel의 `sys.executable`이 수업 `.venv`인지 확인했습니다. [환경 안내](surl:14892)
- [ ] 실습을 저장한 뒤 Kernel을 재시작하고 선행 페이지의 준비 셀부터 위에서 아래로 다시 실행했습니다. 웹 수집은 저장한 자료와 성공·실패 기록을 구분했습니다.

- [ ] 6개 성별×등급 그룹의 인원 표를 만들었습니다.
- [ ] 관찰·근거·추가 질문을 각각 한 문장으로 썼습니다.
- [ ] 그래프의 분모와 결측 처리 기준을 설명했습니다.

## 한글 그래프를 사용할 때

영어 제목·축 이름은 그대로 실행합니다. 한글로 바꾸면 [앞 교시의 설치 글꼴 선택 안내](surl:14897)를 따릅니다. Windows는 Malgun Gothic, Mac은 AppleGothic 등 실제 설치된 글꼴을 사용합니다.

## 파일 위치 확인

자료 폴더는 Windows `C:\kpc-finance-data`, Mac `~/kpc-finance-data`를 사용합니다. `Path.cwd()`가 `raw`를 포함한 최상위 폴더인지 확인합니다. 배포 Notebook의 준비 셀은 이 위치를 찾습니다. 파일명·공백·대소문자와 예제의 CSV 인코딩을 그대로 맞춥니다. [경로와 조작 안내](surl:14881)

## 실행 결과 확인

제공 파일로 실행한 행 수를 기준으로 설명합니다. 라이브 수집 자료나 라이브러리 버전이 달라지면 값이 달라질 수 있습니다.

<!-- period-navigation:start -->
## 교시별 이동

[교시별 전체 안내](surl:14891) · [공식 단원별 안내](surl:14880) · [이전: 10월 7일 1~2교시 — Titanic 시각화와 그래프](surl:14897) · [다음: 10월 7일 4~5교시 — 전처리와 훈련·테스트 분리](surl:14899)
<!-- period-navigation:end -->